# Using RecourseBench

This notebook is a runnable tour of the canonical, user-facing API. Import the
package as `rb` and either:

- call `rb.run(config)` to execute a whole experiment from a config dict, or
- construct components by name from the named namespaces (`rb.datasets`,
  `rb.preprocessors`, `rb.models`, `rb.methods`, `rb.evaluations`) and wire the
  pipeline yourself.

Everything below runs end to end on the bundled `toy_data` dataset — no network,
no GPU, a few seconds total. The two halves produce the *same* metrics, which is
the point: `rb.run()` is just the composed pipeline.

In [3]:
# Install the published package into THIS kernel (%pip, not !pip).
# Pin 0.1.1+: the 0.1.0 build predated the recourse_bench facade.
# --extra-index-url pulls real deps (numpy/pandas/torch) from PyPI;
# TestPyPI does not host them. After this runs, RESTART THE KERNEL.
%pip install -i https://test.pypi.org/simple/ \
    --extra-index-url https://pypi.org/simple \
    "recourse-bench>=0.1.2"

Looking in indexes: https://test.pypi.org/simple/, https://pypi.org/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 49.4 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: recourse-bench
    Found existing installation: recourse_bench 0.1.1
    Uninstalling recourse_bench-0.1.1:
      Successfully uninstalled recourse_bench-0.1.1

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import recourse_bench as rb

print("RecourseBench", getattr(rb, "__version__", "(dev)"))

RecourseBench (dev)


## 1. Discover what is registered

The namespaces are populated dynamically from the registry, so they always
reflect the currently registered components. Use the `list_*` helpers for the
authoritative, up-to-date set of names.

In [ ]:
print("datasets    :", rb.list_datasets())
print("preprocess  :", rb.list_preprocessors())
print("models      :", rb.list_models())
print("methods     :", rb.list_methods())
print("evaluations :", rb.list_evaluations())

datasets    : ['adult', 'adult_cfrl', 'adult_cfvae', 'adult_cogs', 'boston_housing', 'breast_cancer', 'compas', 'compas_carla', 'compas_clue', 'credit', 'credit_cchvae', 'diabetes', 'german', 'german_roar', 'german_sns', 'hepatitis', 'news_popularity', 'synthetic_face', 'toy_data']
preprocess  : ['balance', 'encode', 'finalize', 'reorder', 'scale', 'split']
models      : ['linear', 'mlp', 'mlp_bayesian', 'random_forest', 'sklearn_logistic_regression']
methods     : ['apas', 'arg_ensembling', 'cchvae', 'cemsp', 'cfrl', 'cfvae', 'claproar', 'clue', 'cogs', 'cols', 'cruds', 'cvas_proj', 'dice', 'diverse_dist', 'face', 'feature_tweak', 'gravitational', 'gs', 'larr', 'mace', 'probe', 'proplace', 'rbr', 'revise', 'roar', 'sns', 'toy', 'trex', 'wachter']
evaluations : ['constraints', 'distance', 'examples', 'knn', 'runtime', 'validity', 'ynn']


Each name is an attribute of the matching namespace, and the attribute is the
component *class* — call it to construct an instance. To iterate over components
by name (e.g. for a sweep), use `getattr`.

In [ ]:
# The namespace attribute is the class itself.
print(rb.models.linear)
print(rb.methods.wachter)

# Iterate by name with getattr (the building block for a method sweep).
for name in ["toy", "wachter"]:
    cls = getattr(rb.methods, name)
    print(name, "->", cls.__name__)

<class 'model.linear.linear.LinearModel'>
<class 'method.wachter.wachter.WachterMethod'>
toy -> ToyMethod
wachter -> WachterMethod


## 2. The one-call path: `rb.run(config)`

An experiment is configured as data. The config has four required sections —
`dataset`, `model`, `method`, `evaluation` — plus optional `preprocess`,
`name`, `seed`, `logger`, and `caching`. A single top-level `seed` is propagated
to every component that does not set its own.

`rb.run(config)` is a thin functional facade over `Experiment`; it returns the
metrics table as a one-row `pandas.DataFrame`, with provenance attached under
`metrics.attrs`.

In [ ]:
config = {
    "name": "toy_demo",
    "seed": 7,
    "dataset": {"name": "toy_data"},
    "preprocess": [
        {"name": "scale", "seed": 7, "scaling": "standardize", "range": True},
        {"name": "split", "seed": 7, "split": 0.25, "sample": 4},
    ],
    "model": {
        "name": "linear",
        "seed": 7,
        "device": "cpu",
        "epochs": 30,
        "learning_rate": 0.03,
        "batch_size": 8,
        "optimizer": "adam",
        "criterion": "cross_entropy",
    },
    "method": {
        "name": "toy",
        "seed": 7,
        "device": "cpu",
        "desired_class": 1,
        "max_iterations": 50,
        "step_size": 0.05,
        "lambda_": 0.08,
        "clamp": True,
    },
    "evaluation": [
        {"name": "validity"},
        {"name": "distance"},
    ],
}

metrics = rb.run(config)
metrics

2026-06-30 16:43:15 | INFO | toy_demo | Experiment config:
name: toy_demo
seed: 7
dataset:
  name: toy_data
  seed: 7
preprocess:
- name: scale
  seed: 7
  scaling: standardize
  range: true
- name: split
  seed: 7
  split: 0.25
  sample: 4
- name: finalize
  seed: 7
model:
  name: linear
  seed: 7
  device: cpu
  epochs: 30
  learning_rate: 0.03
  batch_size: 8
  optimizer: adam
  criterion: cross_entropy
method:
  name: toy
  seed: 7
  device: cpu
  desired_class: 1
  max_iterations: 50
  step_size: 0.05
  lambda_: 0.08
  clamp: true
evaluation:
- name: validity
- name: distance



2026-06-30 16:43:15 | INFO | toy_demo | Starting preprocess: ScalePreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Completed preprocess: ScalePreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Starting preprocess: SplitPreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Completed preprocess: SplitPreProcess


...

2026-06-30 16:43:15 | INFO | toy_demo | Starting evaluation: ValidityEvaluation


2026-06-30 16:43:15 | INFO | toy_demo | Completed evaluation: ValidityEvaluation


2026-06-30 16:43:15 | INFO | toy_demo | Starting evaluation: DistanceEvaluation


2026-06-30 16:43:15 | INFO | toy_demo | Completed evaluation: DistanceEvaluation


,validity,distance_l0,distance_l1,distance_l2,distance_linf
0,1.0,4.0,2.830303,1.415152,0.707662


Provenance is attached to the metrics for reproducibility: library version,
config hash, git revision, timestamp, seed, and the resolved component names.

In [ ]:
for k, v in metrics.attrs["provenance"].items():
    print(f"{k:18}: {v}")

library_version   : 0.1.0
config_hash       : 560e66ce475bed2ee048bb28ec2e59d3ffb8ed5c4987c4acc4b8884908804f8a
git_revision      : bf86b1b44721551e07e9daa0ef425cb760fcf9e8
timestamp_utc     : 2026-06-30T16:43:15.317386+00:00
seed              : 7
dataset           : toy_data
model             : linear
method            : toy


### Running from a YAML file

The same config usually lives in a YAML file. `rb.run_config_file(path)` loads
and runs it. The repo ships smoke configs you can run as-is, e.g.:

```python
metrics = rb.run_config_file("experiment/toy/smoke_config.yaml")
```

## 3. The composable path: build the pipeline by hand

When you want to touch the intermediate objects — inspect feature metadata,
look at model probabilities, examine individual counterfactuals — construct the
components yourself. This is exactly what `rb.run()` does internally.

### 3a. Dataset

A dataset is constructed with no required arguments; the raw dataframe and
feature metadata are loaded from bundled offline data. It starts in a *mutable*
state — preprocessing steps mutate it, and the read interface
(`get`, `len`, indexing) only becomes available once it is frozen.

In [ ]:
data = rb.datasets.toy_data()

# Feature metadata is available even while mutable, via attr(<flag>).
print("name          :", data.attr("name"))
print("target column :", data.attr("target_column"))
print("feature order :", data.attr("feature_order"))
print("types         :", data.attr("raw_feature_type"))
print("mutability    :", data.attr("raw_feature_mutability"))
print("actionability :", data.attr("raw_feature_actionability"))

name          : toy_data
target column : approved
feature order : ['age', 'income', 'savings', 'debt', 'credit_score', 'approved']
types         : {'age': 'numerical', 'income': 'numerical', 'savings': 'numerical', 'debt': 'numerical', 'credit_score': 'numerical', 'approved': 'binary'}
mutability    : {'age': False, 'income': True, 'savings': True, 'debt': True, 'credit_score': True, 'approved': False}
actionability : {'age': 'none', 'income': 'same-or-increase', 'savings': 'same-or-increase', 'debt': 'same-or-decrease', 'credit_score': 'same-or-increase', 'approved': 'none'}


### 3b. Preprocessing

Each step's `transform(dataset)` takes a mutable `DatasetObject` and returns the
transformed dataset — or a *tuple* of datasets for steps that split. A typical
pipeline scales, splits into train/test, then `finalize` freezes each split so
the read interface turns on.

In [ ]:
data = rb.datasets.toy_data()
data = rb.preprocessors.scale(seed=7, scaling="standardize", range=True).transform(data)
train_set, test_set = rb.preprocessors.split(seed=7, split=0.25, sample=4).transform(data)

# Freeze each split to enable get()/len()/indexing.
train_set = rb.preprocessors.finalize().transform(train_set)
test_set = rb.preprocessors.finalize().transform(test_set)

print("train rows:", len(train_set), "| test rows:", len(test_set))
print("feature columns:", list(train_set.get(target=False).columns))
train_set.get(target=False).head()

train rows: 18 | test rows: 4
feature columns: ['age', 'income', 'savings', 'debt', 'credit_score']


,age,income,savings,debt,credit_score
9,0.217943,-0.299515,-0.440155,0.186842,-0.282556
16,-0.025342,0.482891,0.498843,-0.782718,0.639593
12,-1.120126,-0.201714,-0.127156,-0.297938,0.072117
0,-1.606696,-1.570923,-1.379153,2.247160,-1.772181
22,1.799297,1.705399,1.750840,-1.146304,1.490807


### 3c. Model

`model.fit(train_set)` trains on a frozen dataset. `predict_proba` returns a
`(n_rows, n_classes)` `torch.Tensor` of probabilities; `predict` returns logits.

In [ ]:
model = rb.models.linear(
    seed=7,
    device="cpu",
    epochs=30,
    learning_rate=0.03,
    batch_size=8,
    optimizer="adam",
    criterion="cross_entropy",
)
model.fit(train_set)

proba = model.predict_proba(test_set)
print("proba shape:", tuple(proba.shape))
print("P(class=1) on test rows:", proba[:, 1].tolist())


linear-fit:   0%|          | 0/30 [00:00<?, ?it/s]


linear-fit:   7%|▋         | 2/30 [00:00<00:01, 19.59it/s]

proba shape: (4, 2)
P(class=1) on test rows: [0.24861961603164673, 0.9429600834846497, 0.002026272239163518, 7.37348455004394e-05]


### 3d. Recourse method

A method wraps the target model. `desired_class` steers which class the
counterfactuals move toward. `method.fit(train_set)` builds any auxiliary search
structures; the inherited `method.predict(test_set)` runs counterfactual search
in batches and returns a *frozen counterfactual dataset* carrying runtime,
prediction, and target-label metadata (failed rows have NaN features and
target `-1`).

In [ ]:
method = rb.methods.toy(
    target_model=model,
    seed=7,
    desired_class=1,
    max_iterations=50,
    step_size=0.05,
    lambda_=0.08,
    clamp=True,
)
method.fit(train_set)

counterfactuals = method.predict(test_set)
print("counterfactual dataset:", type(counterfactuals).__name__, "|", len(counterfactuals), "rows")
counterfactuals.get(target=False)

counterfactual dataset: ToydataDataset | 4 rows


,age,income,savings,debt,credit_score
11,0.582871,0.142756,-0.134759,-0.204471,0.150112
13,-0.876841,-0.055013,0.029344,-0.419133,0.213986
5,-0.511913,-0.098703,-0.121928,0.004985,-0.133106
2,-1.241768,-0.189442,-0.036632,0.333829,-0.302248


We can line up each factual against its counterfactual to see what the method
changed. Note that the immutable feature (`age`) is left untouched.

In [ ]:
factual_X = test_set.get(target=False).reset_index(drop=True)
cf_X = counterfactuals.get(target=False).reset_index(drop=True)

delta = (cf_X - factual_X).round(3)
print("Per-feature change (counterfactual - factual):")
delta

Per-feature change (counterfactual - factual):


,age,income,savings,debt,credit_score
0,0.0,0.149,0.149,-0.149,0.149
1,0.0,0.000,0.000,0.000,0.000
2,-0.0,0.788,0.788,-0.788,0.788
3,-0.0,1.186,1.186,-1.186,1.186


### 3e. Evaluation

Each metric's `evaluate(factuals, counterfactuals)` takes the finalized factual
dataset and the counterfactual dataset and returns a one-row DataFrame of named
metrics. An experiment concatenates these column-wise into the final table.

In [ ]:
import pandas as pd

validity = rb.evaluations.validity().evaluate(test_set, counterfactuals)
distance = rb.evaluations.distance().evaluate(test_set, counterfactuals)

manual_metrics = pd.concat([validity, distance], axis=1)
manual_metrics

,validity,distance_l0,distance_l1,distance_l2,distance_linf
0,1.0,4.0,2.830303,1.415152,0.707662


### The two paths agree

The hand-built pipeline reproduces the metrics from `rb.run()` exactly — same
seed, same components, same numbers.

In [ ]:
common = [c for c in manual_metrics.columns if c in metrics.columns]
comparison = pd.concat(
    [metrics[common].rename(index={0: "rb.run()"}),
     manual_metrics[common].rename(index={0: "manual"})]
)
print("Match:", bool((metrics[common].values == manual_metrics[common].values).all()))
comparison

Match: True


,validity,distance_l0,distance_l1,distance_l2,distance_linf
rb.run(),1.0,4.0,2.830303,1.415152,0.707662
manual,1.0,4.0,2.830303,1.415152,0.707662


## 4. Full control with `Experiment`

`rb.run()` throws away the intermediate artifacts. When you also want the
trained model, the fitted method, the resolved splits, or the generated
counterfactuals, drive the `Experiment` class directly. `run()` returns the
same metrics table, and the artifacts stay available for inspection.

In [ ]:
from experiments import Experiment

exp = Experiment(config)
exp_metrics = exp.run()

print("trained model     :", type(exp.target_model()).__name__)
print("fitted method     :", type(exp.recourse_method()).__name__)
print("train / test rows :", len(exp.train_set()), "/", len(exp.test_set()))
print("counterfactuals   :", type(exp.counterfactuals()).__name__, len(exp.counterfactuals()))
exp_metrics

2026-06-30 16:43:15 | INFO | toy_demo | Experiment config:
name: toy_demo
seed: 7
dataset:
  name: toy_data
  seed: 7
preprocess:
- name: scale
  seed: 7
  scaling: standardize
  range: true
- name: split
  seed: 7
  split: 0.25
  sample: 4
- name: finalize
  seed: 7
model:
  name: linear
  seed: 7
  device: cpu
  epochs: 30
  learning_rate: 0.03
  batch_size: 8
  optimizer: adam
  criterion: cross_entropy
method:
  name: toy
  seed: 7
  device: cpu
  desired_class: 1
  max_iterations: 50
  step_size: 0.05
  lambda_: 0.08
  clamp: true
evaluation:
- name: validity
- name: distance



2026-06-30 16:43:15 | INFO | toy_demo | Starting preprocess: ScalePreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Completed preprocess: ScalePreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Starting preprocess: SplitPreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Completed preprocess: SplitPreProcess


...

2026-06-30 16:43:15 | INFO | toy_demo | Completed evaluation: ValidityEvaluation


2026-06-30 16:43:15 | INFO | toy_demo | Starting evaluation: DistanceEvaluation


2026-06-30 16:43:15 | INFO | toy_demo | Completed evaluation: DistanceEvaluation


trained model     : LinearModel
fitted method     : ToyMethod
train / test rows : 18 / 4
counterfactuals   : ToydataDataset 4


,validity,distance_l0,distance_l1,distance_l2,distance_linf
0,1.0,4.0,2.830303,1.415152,0.707662


## 5. A tiny method sweep

Because components are addressable by name, comparing methods is a loop. Here we
swap only the `method` section of the config and collect the metrics for each.
(`toy` and `wachter` both work on this differentiable linear model.)

In [ ]:
import copy

rows = []
for method_name in ["toy", "wachter"]:
    cfg = copy.deepcopy(config)
    cfg["method"] = {"name": method_name, "seed": 7, "device": "cpu", "desired_class": 1}
    m = rb.run(cfg)
    m.insert(0, "method", method_name)
    rows.append(m)

pd.concat(rows, ignore_index=True)

2026-06-30 16:43:15 | INFO | toy_demo | Experiment config:
name: toy_demo
seed: 7
dataset:
  name: toy_data
  seed: 7
preprocess:
- name: scale
  seed: 7
  scaling: standardize
  range: true
- name: split
  seed: 7
  split: 0.25
  sample: 4
- name: finalize
  seed: 7
model:
  name: linear
  seed: 7
  device: cpu
  epochs: 30
  learning_rate: 0.03
  batch_size: 8
  optimizer: adam
  criterion: cross_entropy
method:
  name: toy
  seed: 7
  device: cpu
  desired_class: 1
evaluation:
- name: validity
- name: distance



2026-06-30 16:43:15 | INFO | toy_demo | Starting preprocess: ScalePreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Completed preprocess: ScalePreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Starting preprocess: SplitPreProcess


2026-06-30 16:43:15 | INFO | toy_demo | Completed preprocess: SplitPreProcess


...

2026-06-30 16:43:16 | INFO | toy_demo | Starting evaluation: ValidityEvaluation


2026-06-30 16:43:16 | INFO | toy_demo | Completed evaluation: ValidityEvaluation


2026-06-30 16:43:16 | INFO | toy_demo | Starting evaluation: DistanceEvaluation


2026-06-30 16:43:16 | INFO | toy_demo | Completed evaluation: DistanceEvaluation


,method,validity,distance_l0,distance_l1,distance_l2,distance_linf
0,toy,1.000000,4.0,2.830875,1.415438,0.707772
1,wachter,0.666667,4.0,4.637656,2.448444,1.447487


## Recap

- **`rb.run(config)` / `rb.run_config_file(path)`** — one call, metrics out, with
  provenance under `metrics.attrs`.
- **Named namespaces** (`rb.datasets`, `rb.preprocessors`, `rb.models`,
  `rb.methods`, `rb.evaluations`) — construct components by registry name; use
  `getattr` to sweep.
- **`list_*()`** — the authoritative set of registered names.
- **`Experiment(config)`** — same run, but keeps the trained model, fitted
  method, resolved splits, and counterfactuals for inspection.

See *Extending the framework* in the docs for the full base-class signatures and
how to register your own datasets, models, methods, and metrics.